# Final replication — Segnon & Trede (Copula-MSM)

Final workflow notebook used to produce the figures, parameter tables, rolling VaR forecasts, and backtesting tables. Reusable code lives in `src/`; this notebook only orchestrates the steps and saves the outputs.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src import (
    RAW_DATA_DIR, PROCESSED_DATA_DIR, REPORTS_DIR, FIGURES_DIR, TABLES_DIR, ensure_project_dirs,
    load_price_csv, log_returns, align_return_frame,
    summary_statistics, format_table_1,
    fit_msm, fit_msm_grid, build_msm_pit_frame,
    fit_garch_marginals, garch_results_table, format_garch_table_3, build_garch_pit_frame, build_garch_volatility_frame,
    fit_copula_grid, copula_results_table, format_copula_table_4,
    SUPPORTED_COPULAS, prepare_bivariate_returns, portfolio_returns,
    forecast_historical_var_rolling, forecast_variance_covariance_var_rolling, forecast_riskmetrics_var_rolling,
    forecast_ccc_garch_var_rolling, forecast_garch_copula_var_rolling, forecast_msm_copula_var_rolling,
    save_var, load_var, concat_var_series,
    make_lr_table, format_lr_table, make_gmm_table, make_spa_table9,
    plot_price_evolution, plot_returns_and_squared_returns, plot_var_forecasts,
)

ensure_project_dirs()
VAR_OUTPUT_DIR = TABLES_DIR / "var_forecasts"
VAR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Prices and returns

The paper studies the NASDAQ Composite and the S&P 500. The returns used below are percentage log returns.

In [ ]:
nasdaq_prices = load_price_csv(RAW_DATA_DIR / "nasdaqcom_yahoo_close.csv")
sp500_prices = load_price_csv(RAW_DATA_DIR / "sp500_yahoo_close.csv")
nasdaq_prices.name = "NASDAQ"
sp500_prices.name = "S&P 500"

prices = pd.concat({"NASDAQ": nasdaq_prices, "S&P 500": sp500_prices}, axis=1).dropna()
prices.index.name = "date"
prices.to_csv(PROCESSED_DATA_DIR / "prices_nasdaq_sp500.csv")
prices.head()

In [ ]:
fig1 = plot_price_evolution(prices, output_path=FIGURES_DIR / "figure_1_prices.html")
fig1.show()

In [ ]:
returns = align_return_frame({
    "NASDAQ": 100 * log_returns(prices["NASDAQ"]),
    "S&P 500": 100 * log_returns(prices["S&P 500"]),
})
returns.to_csv(PROCESSED_DATA_DIR / "returns_nasdaq_sp500.csv")
returns.head()

In [ ]:
fig2 = plot_returns_and_squared_returns(returns, output_path=FIGURES_DIR / "figure_2_returns_squared_returns.html")
fig2.show()

## 2. Descriptive statistics — Table 1

These statistics reproduce the logic of Table 1: mean, standard deviation, skewness, kurtosis, Hurst exponent, tail index, ARCH tests, Jarque-Bera test, and ADF test.

In [ ]:
stats = summary_statistics(returns)
table_1 = format_table_1(stats)
table_1.to_csv(TABLES_DIR / "table_1_statistics_formatted.csv")
table_1

## 3. MSM margins — Table 2

The MSM model is estimated for several values of `k`. For the Copula-MSM VaR part, the paper then fixes the number of MSM components at `k=5`.

In [ ]:
# Long runtime: increase n_starts for the full final version.
msm_table = fit_msm_grid(returns=returns, k_values=range(1, 8), n_starts=30, seed=123)
msm_table.to_csv(TABLES_DIR / "table_2_msm_estimates_raw.csv", index=False)
msm_table

In [ ]:
res_nasdaq_5 = fit_msm(returns["NASDAQ"], k=5, n_starts=30, seed=123)
res_sp500_5 = fit_msm(returns["S&P 500"], k=5, n_starts=30, seed=123)
msm_fit_results_5 = {"NASDAQ": res_nasdaq_5, "S&P 500": res_sp500_5}

pit_msm_5 = build_msm_pit_frame(returns=returns, fit_results=msm_fit_results_5)
pit_msm_5.to_csv(PROCESSED_DATA_DIR / "pit_msm_5.csv")
pit_msm_5.head()

## 4. GARCH margins — Table 3

The Copula-GARCH benchmark uses Gaussian GARCH(1,1) marginal models.

In [ ]:
garch_results = fit_garch_marginals(returns=returns, mean="Constant", dist="normal", rescale=False)
garch_table = garch_results_table(garch_results)
garch_table.to_csv(TABLES_DIR / "table_3_garch_estimates_raw.csv", index=False)

table_3 = format_garch_table_3(garch_table)
table_3.to_csv(TABLES_DIR / "table_3_garch_estimates_formatted.csv")
table_3

In [ ]:
pit_garch = build_garch_pit_frame(garch_results)
pit_garch.to_csv(PROCESSED_DATA_DIR / "pit_garch.csv")

garch_volatility = build_garch_volatility_frame(garch_results)
garch_volatility.to_csv(PROCESSED_DATA_DIR / "garch_conditional_volatility.csv")
pit_garch.head()

## 5. Copulas — Table 4

Copulas are estimated on the MSM and GARCH PIT series. Table 4 compares log-likelihoods, AIC, and BIC.

In [ ]:
pit_msm_5 = pd.read_csv(PROCESSED_DATA_DIR / "pit_msm_5.csv", index_col="date", parse_dates=True)
pit_garch = pd.read_csv(PROCESSED_DATA_DIR / "pit_garch.csv", index_col="date", parse_dates=True)

copula_results = fit_copula_grid({"MSM": pit_msm_5, "GARCH": pit_garch})
copula_table = copula_results_table(copula_results)
copula_table.to_csv(TABLES_DIR / "table_4_copula_estimates_raw.csv", index=False)

table_4 = format_copula_table_4(copula_table)
table_4.to_csv(TABLES_DIR / "table_4_copula_estimates_formatted.csv", index=False)
table_4

## 6. Rolling VaR forecasts

The backtesting methodology follows a fixed rolling-window scheme: 1135 in-sample observations, 500 one-step-ahead forecasts, and VaR levels of 5% and 1%. VaR is stored as a signed return quantile, so a violation is `r_p,t < VaR_t`.

In [ ]:
PI = 0.5
WEIGHTS = np.array([PI, 1.0 - PI])
WINDOW_SIZE = 1135
N_OOS = 500
ALPHA_5 = 0.05
ALPHA_1 = 0.01

returns_var = prepare_bivariate_returns(returns, n_insample=WINDOW_SIZE, n_oos=N_OOS)
portfolio_ret = portfolio_returns(returns_var, weights=WEIGHTS)
oos_index = returns_var.index[WINDOW_SIZE:WINDOW_SIZE + N_OOS]
portfolio_ret_oos = portfolio_ret.loc[oos_index]
portfolio_ret_oos.head()

### 6.1 Fast benchmark models

These models are relatively inexpensive to run: Historical Simulation, Variance-Covariance, RiskMetrics, and CCC-GARCH.

In [ ]:
def compute_benchmark_vars(alpha: float, suffix: str):
    outputs = {
        "Historical": forecast_historical_var_rolling(returns_var, alpha, WEIGHTS, WINDOW_SIZE, N_OOS),
        "Variance-Covariance": forecast_variance_covariance_var_rolling(returns_var, alpha, WEIGHTS, WINDOW_SIZE, N_OOS, include_mean=True),
        "RiskMetrics": forecast_riskmetrics_var_rolling(returns_var, alpha, WEIGHTS, 0.94, WINDOW_SIZE, N_OOS, include_mean=False),
        "CCC-GARCH": forecast_ccc_garch_var_rolling(returns_var, alpha, WEIGHTS, WINDOW_SIZE, N_OOS, include_mean=True, verbose=True),
    }
    filenames = {"Historical": "hist", "Variance-Covariance": "vc", "RiskMetrics": "rm", "CCC-GARCH": "ccc"}
    for name, series in outputs.items():
        save_var(series.rename(name), VAR_OUTPUT_DIR, f"var_{filenames[name]}_{suffix}.csv")
    return outputs

bench_5 = compute_benchmark_vars(ALPHA_5, "5")
bench_1 = compute_benchmark_vars(ALPHA_1, "1")

### 6.2 Copula-GARCH

The loop below replaces repeated copula-by-copula cells.

In [ ]:
for alpha, suffix in [(ALPHA_5, "5"), (ALPHA_1, "1")]:
    for copula in SUPPORTED_COPULAS:
        series = forecast_garch_copula_var_rolling(
            returns=returns_var,
            copula=copula,
            alpha=alpha,
            weights=WEIGHTS,
            n_insample=WINDOW_SIZE,
            n_oos=N_OOS,
            integration_nodes=501,
            root_tol=1e-4,
            verbose=True,
        )
        save_var(series.rename(f"Copula-GARCH {copula}"), VAR_OUTPUT_DIR, f"var_garch_{copula}_{suffix}.csv")

### 6.3 Copula-MSM

This is the most computationally expensive step. For a full replication, run all copulas. For a faster version, restrict `MSM_COPULAS_TO_RUN` to `student` and `gaussian`, then load the other saved VaR forecasts if they are already available.

In [ ]:
MSM_COPULAS_TO_RUN = ["student", "gaussian", "plackett", "clayton"]

for alpha, suffix in [(ALPHA_5, "5"), (ALPHA_1, "1")]:
    for copula in MSM_COPULAS_TO_RUN:
        series = forecast_msm_copula_var_rolling(
            returns=returns_var,
            copula=copula,
            alpha=alpha,
            weights=WEIGHTS,
            k=5,
            n_insample=WINDOW_SIZE,
            n_oos=N_OOS,
            n_starts=10,
            seed=123,
            integration_nodes=501,
            root_tol=1e-4,
            verbose=True,
        )
        save_var(series.rename(f"Copula-MSM {copula}"), VAR_OUTPUT_DIR, f"var_msm_{copula}_{suffix}.csv")

### 6.4 Build VaR panels

In [ ]:
def build_var_panel(alpha_suffix: str, include_msm=("student", "gaussian", "plackett", "clayton")):
    series = {
        "Historical": load_var(VAR_OUTPUT_DIR, f"var_hist_{alpha_suffix}.csv"),
        "Variance-Covariance": load_var(VAR_OUTPUT_DIR, f"var_vc_{alpha_suffix}.csv"),
        "RiskMetrics": load_var(VAR_OUTPUT_DIR, f"var_rm_{alpha_suffix}.csv"),
        "CCC-GARCH": load_var(VAR_OUTPUT_DIR, f"var_ccc_{alpha_suffix}.csv"),
    }
    for copula in SUPPORTED_COPULAS:
        label = copula.replace("_", " ").title().replace("Sjc", "SJC")
        series[f"Copula-GARCH {label}"] = load_var(VAR_OUTPUT_DIR, f"var_garch_{copula}_{alpha_suffix}.csv")
    for copula in include_msm:
        label = copula.replace("_", " ").title().replace("Sjc", "SJC")
        series[f"Copula-MSM {label}"] = load_var(VAR_OUTPUT_DIR, f"var_msm_{copula}_{alpha_suffix}.csv")
    return concat_var_series(series, expected_length=N_OOS)

var_panel_5 = build_var_panel("5")
var_panel_1 = build_var_panel("1", include_msm=("student", "gaussian"))
var_panel_5.to_csv(VAR_OUTPUT_DIR / "var_panel_5pct.csv")
var_panel_1.to_csv(VAR_OUTPUT_DIR / "var_panel_1pct.csv")
var_panel_5.head()

In [ ]:
fig3 = plot_var_forecasts(
    portfolio_returns=portfolio_ret_oos.rename("Portfolio returns"),
    var_forecasts=var_panel_5[["Historical", "Variance-Covariance", "RiskMetrics", "CCC-GARCH", "Copula-GARCH Student", "Copula-MSM Student"]],
    output_path=FIGURES_DIR / "figure_3_var_5pct.html",
    title="Figure 3 — VaR forecasts at the 5% confidence level",
)
fig3.show()

In [ ]:
fig4 = plot_var_forecasts(
    portfolio_returns=portfolio_ret_oos.rename("Portfolio returns"),
    var_forecasts=var_panel_1[["Historical", "Variance-Covariance", "RiskMetrics", "CCC-GARCH", "Copula-GARCH Student", "Copula-MSM Student"]],
    output_path=FIGURES_DIR / "figure_4_var_1pct.html",
    title="Figure 4 — VaR forecasts at the 1% confidence level",
)
fig4.show()

## 7. LR backtesting — Tables 5 and 6

Christoffersen's tests summarize: empirical frequency of violations (`EFV`), unconditional coverage (`uc`), independence (`ind`), and conditional coverage (`cc`).

In [ ]:
table_5_raw = make_lr_table(returns_var, var_panel_5, alpha=ALPHA_5, weights=WEIGHTS)
table_5 = format_lr_table(table_5_raw, msm_submodels=["Student", "Normal", "Plackett", "Clayton"])
table_5_raw.to_csv(TABLES_DIR / "table_5_lr_var_5pct_raw.csv")
table_5.to_csv(TABLES_DIR / "table_5_lr_var_5pct_formatted.csv", encoding="utf-8-sig")
table_5.to_latex(TABLES_DIR / "table_5_lr_var_5pct.tex", multicolumn=True, multicolumn_format="c", escape=False,
                 caption="The results of LR test using VaR(5%) forecasts.", label="tab:lr_var_5")
table_5

In [ ]:
table_6_raw = make_lr_table(returns_var, var_panel_1, alpha=ALPHA_1, weights=WEIGHTS)
table_6 = format_lr_table(table_6_raw, msm_submodels=["Student", "Normal"])
table_6_raw.to_csv(TABLES_DIR / "table_6_lr_var_1pct_raw.csv")
table_6.to_csv(TABLES_DIR / "table_6_lr_var_1pct_formatted.csv", encoding="utf-8-sig")
table_6.to_latex(TABLES_DIR / "table_6_lr_var_1pct.tex", multicolumn=True, multicolumn_format="c", escape=False,
                 caption="The results of LR test using VaR(1%) forecasts.", label="tab:lr_var_1")
table_6

## 8. GMM duration-based tests — Tables 7 and 8

These tests use the durations between VaR violations. Rows `Juc`, `Jcc`, and `Jind` correspond to the UC, CC, and IND hypotheses with different numbers of moment conditions.

In [ ]:
table_7 = make_gmm_table(returns_var, var_panel_5, alpha=ALPHA_5, weights=WEIGHTS)
table_7.to_csv(TABLES_DIR / "table_7_gmm_var_5pct_raw.csv")
table_7

In [ ]:
table_8 = make_gmm_table(returns_var, var_panel_1, alpha=ALPHA_1, weights=WEIGHTS)
table_8.to_csv(TABLES_DIR / "table_8_gmm_var_1pct_raw.csv")
table_8

## 9. SPA test — Table 9

Each model is taken in turn as the basis model. VaR loss and smooth VaR loss are computed at the 5% and 1% levels.

In [ ]:
table_9 = make_spa_table9(
    portfolio_returns=portfolio_ret,
    var_5_panel=var_panel_5,
    var_1_panel=var_panel_1,
    B=5000,
    block_p=0.1,
    nu=25.0,
    seed=123,
)
table_9_formatted = table_9.map(lambda x: f"{x:.3f}")
table_9_formatted.to_csv(TABLES_DIR / "table_9_spa_formatted.csv")
table_9_formatted